# CAZ Implementation Demo

*From formulas to Paper 4: implement the metrics and reproduce the cross-architecture convergence result*

This notebook works through the mathematics behind the CAZ framework:

1. **The three metrics** — Fisher separation, coherence, and velocity, implemented from first principles and verified against stored results
2. **Procrustes alignment** — the rotation that maps concept directions across architectures, implemented and demonstrated on real data
3. **The PRH result** — a live re-run of the Paper 4 analysis using pre-computed CAZ results downloaded from Hugging Face, reproducing the matched vs. mismatched cosine distribution

No model loading or GPU required. All data comes from the [Rosetta Activations](https://huggingface.co/datasets/james-ra-henry/Rosetta-Activations) dataset.

**[github.com/jamesrahenry/Rosetta](https://github.com/jamesrahenry/Rosetta)** · Henry (2026a, 2026d)

In [ ]:
%pip install -q "rosetta_tools>=1.2.0" huggingface_hub matplotlib numpy scipy pandas

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.linalg import orthogonal_procrustes
from scipy.stats import mannwhitneyu
from huggingface_hub import hf_hub_download
from pathlib import Path

HF_REPO = "james-ra-henry/Rosetta-Activations"
print("Imports ready.")

## 1. Fisher separation

$S(l)$ measures how separable the two class representations are at layer $l$.

$$S(l) = \frac{\|\mu_+ - \mu_-\|}{\sqrt{\frac{1}{2}(\sigma_+^2 + \sigma_-^2)}}$$

$\mu_+, \mu_-$ are the class centroids; $\sigma_+^2, \sigma_-^2$ are within-class variances projected onto the concept direction $\hat{d} = (\mu_+ - \mu_-)/\|\mu_+ - \mu_-\|$. This is the Fisher discriminant ratio — a natural measure that normalises for within-class scatter.

In [ ]:
def separation(pos: np.ndarray, neg: np.ndarray) -> float:
    mu_p, mu_n = pos.mean(0), neg.mean(0)
    diff = mu_p - mu_n
    norm = np.linalg.norm(diff)
    if norm < 1e-12:
        return 0.0
    d_hat = diff / norm
    var_p = ((pos @ d_hat) - (pos @ d_hat).mean()) ** 2
    var_n = ((neg @ d_hat) - (neg @ d_hat).mean()) ** 2
    within = np.sqrt(0.5 * (var_p.mean() + var_n.mean()))
    return float(norm / within) if within > 1e-12 else float(norm)

# Sanity check
rng = np.random.default_rng(42)
pos_sep = rng.normal([2, 0], 0.4, (30, 2))
neg_sep = rng.normal([0, 0], 0.4, (30, 2))
pos_mix = rng.normal([0, 0], 1.5, (30, 2))
neg_mix = rng.normal([0, 0], 1.5, (30, 2))
print(f"Separation (well-separated classes):  {separation(pos_sep, neg_sep):.3f}")
print(f"Separation (overlapping classes):     {separation(pos_mix, neg_mix):.3f}")

## 2. Coherence

$C(l)$ measures how geometrically organised each class is — whether positive examples all point in a consistent direction, not just that the centroids are far apart.

$$C(l) = \frac{1}{2}\left(\frac{1}{n_+}\sum_i \frac{x_i^+ \cdot \mu_+}{\|x_i^+\| \|\mu_+\|} + \frac{1}{n_-}\sum_j \frac{x_j^- \cdot \mu_-}{\|x_j^-\| \|\mu_-\|}\right)$$

A high-separation, low-coherence layer is a diffuse separation — the centroids are far apart but individual examples are scattered. A CAZ peak typically has both high $S$ and high $C$.

In [ ]:
def coherence(pos: np.ndarray, neg: np.ndarray) -> float:
    def _class_coh(acts: np.ndarray) -> float:
        centroid = acts.mean(0)
        c_norm = np.linalg.norm(centroid)
        if c_norm < 1e-12:
            return 0.0
        norms = np.linalg.norm(acts, axis=1, keepdims=True).clip(min=1e-12)
        return float(((acts / norms) @ (centroid / c_norm)).mean())
    return 0.5 * (_class_coh(pos) + _class_coh(neg))

## 3. Velocity — and verifying against stored results

$v(l)$ is the layer-wise derivative of $S(l)$: the rate at which separation is changing. It peaks *before* the separation peak, marking the onset of rapid concept construction.

$$v(l) \approx \frac{S(l + w/2) - S(l - w/2)}{w}$$

The stored CAZ JSON contains pre-computed values for all three metrics. We verify our velocity implementation by comparing it to the stored values.

In [ ]:
def velocity(separations: list, window: int = 3) -> list:
    s = np.array(separations, float)
    pad = window // 2
    s_pad = np.pad(s, pad, mode="edge")
    return [(s_pad[i + window] - s_pad[i]) / window for i in range(len(s))]

# Download one CAZ file and verify
path = hf_hub_download(
    HF_REPO,
    filename="models/Qwen_Qwen2.5_7B_Instruct/caz_causation.json",
    repo_type="dataset",
)
with open(path) as f:
    caz_ref = json.load(f)

stored_metrics = caz_ref["layer_data"]["metrics"]
stored_sep = [m["separation_fisher"] for m in stored_metrics]
stored_vel = [m["velocity"] for m in stored_metrics]
our_vel = velocity(stored_sep, window=3)

fig, ax = plt.subplots(figsize=(9, 3))
layers = [m["layer"] for m in stored_metrics]
ax.plot(layers, stored_vel, "b-", lw=2, label="stored (rosetta_tools)")
ax.plot(layers, our_vel, "r--", lw=1.5, alpha=0.8, label="our implementation")
ax.set_xlabel("Layer")
ax.set_ylabel("Velocity")
ax.set_title("Velocity verification — causation, Qwen2.5-7B-Instruct")
ax.legend()
ax.grid(alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()
print(f"Max absolute difference: {max(abs(a-b) for a,b in zip(stored_vel, our_vel)):.2e}")

## 4. Procrustes alignment

The central question of Paper 4: do different transformer architectures encode concepts in the same geometric directions? If so, we'd expect a single rotation to map one model's concept space onto another's.

The **Orthogonal Procrustes problem** finds exactly this rotation:

$$\min_{R:\, R^T R = I} \| A R - B \|_F \quad \Rightarrow \quad R = V U^T,\quad U \Sigma V^T = \text{SVD}(A^T B)$$

Given calibration activations $A$ (model 1) and $B$ (model 2), $R$ is the rotation that best maps $B$'s coordinate frame into $A$'s.

In [ ]:
def procrustes_rotation(source: np.ndarray, target: np.ndarray) -> np.ndarray:
    """Find R s.t. target @ R ≈ source."""
    A = np.asarray(source, float)
    B = np.asarray(target, float)
    A -= A.mean(0); B -= B.mean(0)
    R, _ = orthogonal_procrustes(B, A)  # scipy: finds R minimising ||B@R - A||_F
    return R

def cosine(v1: np.ndarray, v2: np.ndarray) -> float:
    v1, v2 = np.asarray(v1, float), np.asarray(v2, float)
    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-12))

# ── Toy demonstration: known 45° rotation ────────────────────────────────────
rng = np.random.default_rng(0)
theta = np.radians(45)
R_true = np.array([[np.cos(theta), -np.sin(theta)],
                   [np.sin(theta),  np.cos(theta)]])

# Model A: standard frame
acts_A = rng.normal(0, 1, (60, 2))
true_dir_A = np.array([1.0, 0.0])

# Model B: same concept, coordinate frame rotated 45°
acts_B = acts_A @ R_true
true_dir_B = true_dir_A @ R_true

# Without alignment
print(f"Raw cosine (pre-alignment):      {cosine(true_dir_A, true_dir_B):.4f}  (cos 45° = {np.cos(theta):.4f})")

# Recover rotation and align
R_hat = procrustes_rotation(acts_A, acts_B)
dir_B_aligned = true_dir_B @ R_hat
print(f"Aligned cosine (post-rotation):  {cosine(true_dir_A, dir_B_aligned):.4f}  (expected: 1.0)")

## 5. The PRH question: do different architectures share a geometric language?

Paper 4 tests this with a depth-stratified protocol:

1. For each pair of same-dimension models $(A, B)$, extract the dominant concept direction at three proportional depths: 30%, 50%, 70% of total layers
2. **LOCO fit**: hold out one concept; use the directions for the remaining 6 concepts as calibration to fit a Procrustes rotation
3. **Test**: apply the rotation to the held-out concept's direction and measure cosine similarity to the target model's direction
4. Compare **matched** (same concept, different model) vs **mismatched** (different concept, different model) cosines

We demonstrate this on `Qwen2.5-7B` vs `Gemma-2-9B` — genuinely different architectures (Qwen GQA vs Gemma alternating attention), different training, different companies, but the same hidden dimension (3584). The dominant direction at each layer is stored in the CAZ JSON as `dom_vector`, so this runs entirely from downloaded data.

In [ ]:
# Qwen2.5-7B vs Gemma-2-9B: genuinely cross-architecture, same hidden dimension
# Qwen uses GQA; Gemma-2 uses alternating local/global attention — different families entirely
PAIR = {
    "model_a": "Qwen/Qwen2.5-7B",      # 28L, 3584-dim, GQA
    "model_b": "google/gemma-2-9b",     # 42L, 3584-dim, alternating attention
}
TEST_CONCEPTS = ["credibility", "certainty", "causation",
                 "temporal_order", "negation", "sentiment", "moral_valence"]
DEPTHS = [0.3, 0.5, 0.7]

def model_key(model_id: str) -> str:
    return model_id.replace("/", "_").replace("-", "_")

def get_dom_vector_at_depth(caz_json: dict, target_depth: float) -> np.ndarray:
    """Extract the dominant concept direction at a given proportional depth."""
    metrics = caz_json["layer_data"]["metrics"]
    n_layers = caz_json["n_layers"]
    target_layer = round(target_depth * n_layers)
    closest = min(metrics, key=lambda m: abs(m["layer"] - target_layer))
    return np.array(closest["dom_vector"], dtype=np.float64)

# Download all concept files for both models
directions = {"model_a": {}, "model_b": {}}
for role, model_id in PAIR.items():
    key = model_key(model_id)
    for concept in TEST_CONCEPTS:
        path = hf_hub_download(
            HF_REPO,
            filename=f"models/{key}/caz_{concept}.json",
            repo_type="dataset",
        )
        with open(path) as f:
            caz_data = json.load(f)
        directions[role][concept] = {
            d: get_dom_vector_at_depth(caz_data, d) for d in DEPTHS
        }

print(f"Downloaded directions for {len(TEST_CONCEPTS)} concepts × 2 models × {len(DEPTHS)} depths")
print(f"Model A: {PAIR['model_a']}  ({caz_data['n_layers']}L, {caz_data['hidden_dim']}-dim)")
print(f"Model B: {PAIR['model_b']}")

In [ ]:
# ── Alternative cross-architecture pairs ─────────────────────────────────────
# All pairs below are same hidden-dim (no PCA projection needed) and cross-family.
# Swap any into PAIR above — the LOCO test code is architecture-agnostic.
#
# SMALL (120–400M, 768-dim) — fast download, good for quick verification
#   PAIR = {"model_a": "openai-community/gpt2",    "model_b": "EleutherAI/pythia-160m"}
#   PAIR = {"model_a": "openai-community/gpt2",    "model_b": "facebook/opt-125m"}
#   PAIR = {"model_a": "EleutherAI/pythia-160m",   "model_b": "facebook/opt-125m"}
#
# MEDIUM (2–3B)
#   # Pythia vs OPT, both 2560-dim, MHA — well-studied pair
#   PAIR = {"model_a": "EleutherAI/pythia-2.8b",   "model_b": "facebook/opt-2.7b"}
#
#   # Pythia vs Phi-2 — EleutherAI vs Microsoft, same dim
#   PAIR = {"model_a": "EleutherAI/pythia-2.8b",   "model_b": "microsoft/phi-2"}
#
#   # Qwen vs Llama — Alibaba vs Meta, both 2048-dim
#   PAIR = {"model_a": "Qwen/Qwen2.5-3B",          "model_b": "meta-llama/Llama-3.2-1B"}
#
# LARGE (7–9B)  ← DEFAULT
#   # Qwen2.5-7B vs Gemma-2-9B — GQA vs alternating attention, both 3584-dim
#   PAIR = {"model_a": "Qwen/Qwen2.5-7B",          "model_b": "google/gemma-2-9b"}  ← DEFAULT
#
#   # Pythia-6.9B vs OPT-6.7B — both 4096-dim, different training
#   PAIR = {"model_a": "EleutherAI/pythia-6.9b",   "model_b": "facebook/opt-6.7b"}
#
#   # Llama-3.1-8B vs Mistral-7B — both 4096-dim, Meta vs Mistral
#   PAIR = {"model_a": "meta-llama/Llama-3.1-8B",  "model_b": "mistralai/Mistral-7B-v0.3"}
#
#   # Llama-3.1-8B vs Pythia-6.9B — 4096-dim, transformer research vs instruction-focused
#   PAIR = {"model_a": "meta-llama/Llama-3.1-8B",  "model_b": "EleutherAI/pythia-6.9b"}
#
# VERY LARGE (12B+)
#   # Pythia-12B vs Qwen2.5-14B — both 5120-dim, cross-family
#   PAIR = {"model_a": "EleutherAI/pythia-12b",    "model_b": "Qwen/Qwen2.5-14B"}
#
#   # Llama-3.1-70B vs Qwen2.5-72B — both 8192-dim, Meta vs Alibaba at scale
#   PAIR = {"model_a": "meta-llama/Llama-3.1-70B", "model_b": "Qwen/Qwen2.5-72B"}

In [ ]:
# Run LOCO test: hold out each concept in turn, fit rotation on the rest
results = []
for held_concept in TEST_CONCEPTS:
    fit_concepts = [c for c in TEST_CONCEPTS if c != held_concept]

    # Build calibration matrices: [n_fit_concepts × n_depths, hidden_dim]
    cal_a = np.vstack([directions["model_a"][c][d] for c in fit_concepts for d in DEPTHS])
    cal_b = np.vstack([directions["model_b"][c][d] for c in fit_concepts for d in DEPTHS])

    # Fit Procrustes rotation from model_b → model_a
    R = procrustes_rotation(cal_a, cal_b)

    # Evaluate on held-out concept: matched cosine
    matched_cosines = []
    for d in DEPTHS:
        dir_a = directions["model_a"][held_concept][d]
        dir_b = directions["model_b"][held_concept][d]
        dir_b_aligned = dir_b @ R
        matched_cosines.append(cosine(dir_a, dir_b_aligned))

    # Mismatched: align other concepts' directions and compare to held concept in model_a
    mismatched_cosines = []
    for other_concept in fit_concepts:
        for d in DEPTHS:
            dir_a_held = directions["model_a"][held_concept][d]
            dir_b_other = directions["model_b"][other_concept][d]
            dir_b_other_aligned = dir_b_other @ R
            mismatched_cosines.append(cosine(dir_a_held, dir_b_other_aligned))

    results.append({
        "concept": held_concept,
        "matched_mean": np.mean(matched_cosines),
        "mismatched_mean": np.mean(mismatched_cosines),
        "delta": np.mean(matched_cosines) - np.mean(mismatched_cosines),
    })

print(f"{'Concept':20s}  {'Matched':>9s}  {'Mismatched':>11s}  {'Δ':>7s}")
print("-" * 55)
for r in results:
    sign = "+" if r["delta"] > 0 else "-"
    print(f"{r['concept']:20s}  {r['matched_mean']:9.4f}  {r['mismatched_mean']:11.4f}  {sign}{abs(r['delta']):.4f}")

n_pos = sum(1 for r in results if r["delta"] > 0)
print(f"\nPositive Δ: {n_pos}/{len(results)}")
print(f"Grand means — matched: {np.mean([r['matched_mean'] for r in results]):.4f}  "
      f"mismatched: {np.mean([r['mismatched_mean'] for r in results]):.4f}")

## 6. Reproduce Paper 4: full 563-pair result

The two-model demo above illustrates the protocol. Paper 4 runs this across all same-dimension model pairs in the dataset — 563 observations, 17 concepts, multiple architecture families.

The pre-computed results are bundled with this repository in `data/prh_p5_samedim_n250.json`. We load them and reproduce the matched vs. mismatched distribution figure.

In [ ]:
# Load pre-computed p5 results
data_path = Path("../data/prh_p5_samedim_n250.json")
if not data_path.exists():
    # Fallback: try current directory (e.g. when running from repo root)
    data_path = Path("data/prh_p5_samedim_n250.json")

with open(data_path) as f:
    p5 = json.load(f)

summary = p5["summary"]["grand"]
pair_results = p5["pair_results"]

print(f"N observations:    {summary['n_observations']}")
print(f"N positive delta:  {summary['n_positive_delta']} / {summary['n_observations']}")
print(f"Mean matched:      {summary['mean_matched']:.4f}")
print(f"Mean mismatched:   {summary['mean_mismatched']:.4f}")
print(f"Mean delta:        {summary['mean_delta']:.4f}")
print(f"Mann-Whitney p:    {summary['mannwhitney_p']:.2e}")
print(f"Bootstrap 95% CI:  [{summary['bootstrap_ci_95'][0]:.4f}, {summary['bootstrap_ci_95'][1]:.4f}]")

In [ ]:
# Extract all matched and mismatched cosines
all_matched, all_mismatched, all_deltas = [], [], []
for pr in pair_results:
    cos_matrix = np.array(pr["cos_matrix"])  # [n_concepts, n_depths]
    # Diagonal = matched (same concept in model_a and model_b)
    for i, concept in enumerate(pr["fit_concepts"]):
        matched_cos = cos_matrix[i]  # shape [n_depths]
        all_matched.extend(matched_cos.tolist())
        for j in range(len(pr["fit_concepts"])):
            if j != i:
                all_mismatched.extend(cos_matrix[j].tolist())
    all_deltas.append(pr["obs_delta"])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Paper 4 — Depth-stratified PRH result (N=563 model pairs)",
             fontsize=13, fontweight="bold")

# Left: matched vs mismatched cosine distributions
ax = axes[0]
bins = np.linspace(-0.1, 1.05, 40)
ax.hist(all_mismatched, bins=bins, alpha=0.6, color="#90CAF9", label=f"Mismatched  (mean {summary['mean_mismatched']:.3f})")
ax.hist(all_matched, bins=bins, alpha=0.7, color="#1565C0", label=f"Matched       (mean {summary['mean_matched']:.3f})")
ax.axvline(summary["mean_matched"], color="#1565C0", lw=2, ls="--")
ax.axvline(summary["mean_mismatched"], color="#90CAF9", lw=2, ls="--")
ax.set_xlabel("Cosine similarity (post-alignment)", fontsize=11)
ax.set_ylabel("Count", fontsize=11)
ax.set_title("Cosine distributions", fontsize=11)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)

# Right: per-concept mean delta
byconcept = p5["summary"]["by_concept"]
concept_names = sorted(byconcept, key=lambda c: byconcept[c]["mean_delta"])
delta_vals = [byconcept[c]["mean_delta"] for c in concept_names]
pval_marker = ["*" if byconcept[c]["mannwhitney_p"] < 0.001 else "" for c in concept_names]

ax = axes[1]
bars = ax.barh(concept_names, delta_vals,
               color=["#1565C0" if d > 0 else "#C62828" for d in delta_vals],
               height=0.65, alpha=0.85)
for bar, marker in zip(bars, pval_marker):
    if marker:
        ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
                marker, va="center", fontsize=10, color="#1565C0")
ax.axvline(0, color="#424242", lw=0.8)
ax.set_xlabel("Mean Δ (matched − mismatched cosine)", fontsize=11)
ax.set_title("Per-concept Δ  (* p < 0.001)", fontsize=11)
ax.grid(axis="x", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

print(f"\nAll {summary['n_positive_delta']}/{summary['n_observations']} observations show positive delta (matched > mismatched).")

## Summary

From first principles to a published result:

| Step | What we built |
|------|---------------|
| Fisher separation | Centroid distance normalised by within-class scatter |
| Coherence | Mean cosine similarity to class centroid — directional consistency |
| Velocity | Finite-difference derivative — rate of concept construction |
| Procrustes rotation | SVD-based orthogonal alignment between activation spaces |
| LOCO test | Hold-out evaluation: fit rotation on 6 concepts, test on the 7th |
| Paper 4 result | 563/563 model pairs show matched > mismatched cosine; p = 4.8 × 10⁻²²⁹ |

The result says: after a single orthogonal rotation, transformer models consistently encode the same concepts in more geometrically similar directions than different concepts — across architectures, scales, and training regimes. This is direct evidence for the Platonic Representation Hypothesis at the level of concept geometry.

---

| Paper | |
|-------|-|
| Paper 1 — CAZ Framework | [Henry 2026a](https://arxiv.org/abs/PLACEHOLDER) |
| Paper 4 — Cross-architecture PRH | [Henry 2026d](https://arxiv.org/abs/PLACEHOLDER) |
| `rosetta_tools` | [GitHub](https://github.com/jamesrahenry/Rosetta_Tools) |
| Concept pairs dataset | [Rosetta_Concept_Pairs](https://github.com/jamesrahenry/Rosetta_Concept_Pairs) |